In [ ]:
# STEP 0 (run once): install the SDKs
# WHY (tech): Installs each provider's Python SDK, which handles HTTP requests, authentication and response parsing.
# ELI5: Download a translator app for each AI company.
!pip install -q -U google-genai openai anthropic
# If you later get an ImportError: Runtime -> Restart session, then run the cells again (skip this one).

Gemini CHATBOT

In [ ]:
# WHY (tech): Imports Google's Gen AI SDK, which provides the Client class for calling Gemini.
# ELI5: Bring in Google's phone.
from google import genai
# WHY (tech): Imports typed config objects for settings like temperature and system instructions.
# ELI5: Get Google's order form with boxes for settings.
from google.genai import types
# WHY (tech): Imports Colab's secrets reader so the API key is never written in the code.
# ELI5: Get the key to your locked password drawer.
from google.colab import userdata

# WHY (tech): Creates an authenticated client once; every later Gemini call reuses this connection.
# ELI5: Call Google and show your ID card once.
gemini = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
# WHY (tech): Stores the model ID in one variable so later cells can reuse or change it easily.
# ELI5: Write the brain's name on a sticky note.
MODEL = "gemini-3.8-flash"
# WHY (tech): A tiny test request confirms the key and model work before doing real tasks.
# ELI5: Say "hello" once to check the phone line works.
test = gemini.models.generate_content(model=MODEL, contents="Say OK")
# WHY (tech): .text pulls only the answer text out of the full response object.
# ELI5: Read the reply to confirm you're connected.
print("✅ Gemini connected:", test.text)

In [ ]:
# SINGLE QUESTION (no memory)
# WHY (tech): One stateless request: the model sees only this prompt, with no earlier conversation.
# ELI5: Ask one question to someone who forgets you straight after.
response = gemini.models.generate_content(
    # WHY (tech): Reuses the model ID set in the connection cell.
    # ELI5: Use the brain named on the sticky note.
    model=MODEL,
    # WHY (tech): The input text Gemini reads and responds to.
    # ELI5: Your question.
    contents="Explain what an AI agent is in 2 lines",
)

# WHY (tech): Extracts and prints only the generated answer text.
# ELI5: Show the answer.
print(response.text)

In [ ]:
# CHAT WITH MEMORY - turn 1
# WHY (tech): chats.create builds a chat session object that stores message history automatically.
# ELI5: Open a notebook that Gemini writes the whole conversation into.
chat = gemini.chats.create(model=MODEL)

# WHY (tech): send_message adds your text to the session history, sends it, and saves the reply.
# ELI5: Say something; Gemini answers and writes both down.
response1 = chat.send_message("Hi! My name is Satvik and I teach AI.")

# WHY (tech): Prints the answer text from the first turn.
# ELI5: Show Gemini's reply.
print(response1.text)

In [ ]:
# CHAT WITH MEMORY - turn 2
# WHY (tech): Same chat object, so earlier turns are resent automatically and Gemini can recall them.
# ELI5: Gemini rereads the notebook, so it remembers your name.
response2 = chat.send_message("What's my name, and what do I do?")

# WHY (tech): Prints the second answer, which should use facts from turn 1.
# ELI5: Show the reply that proves it remembered.
print(response2.text)

OPENAI CHATBOT

In [ ]:
# Importing libraries, setting the model and testing the connection
# WHY (tech): Imports OpenAI's SDK; Groq and others also accept this same request format.
# ELI5: Bring in OpenAI's phone.
from openai import OpenAI
# WHY (tech): Imports Colab's secrets reader so the key stays out of the code.
# ELI5: Get the key to your password drawer.
from google.colab import userdata

# WHY (tech): Authenticated client using OpenAI's default endpoint; reused by every later OpenAI cell.
# ELI5: Call OpenAI and show your ID card.
openai_client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
# WHY (tech): Model ID in one variable; overwrites the Gemini MODEL value set earlier.
# ELI5: Write the GPT brain's name on the sticky note.
MODEL = "gpt-6.1-sol"
# WHY (tech): Tiny test request confirms key, credit and model are all working.
# ELI5: Say "hello" once to check the line works.
test = openai_client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say OK"}])
# WHY (tech): OpenAI returns a list of choices; we read the first one's message text.
# ELI5: Read the reply to confirm you're connected.
print("✅ OpenAI connected:", test.choices[0].message.content)

In [ ]:
# SINGLE QUESTION (no memory)
# WHY (tech): Chat Completions call with one user message; stateless, so nothing is remembered afterwards.
# ELI5: Ask one question to someone who forgets you straight after.
response = openai_client.chat.completions.create(
    # WHY (tech): Which GPT model processes the request.
    # ELI5: Which brain answers.
    model=MODEL,
    # WHY (tech): Message list with a single user turn in role/content format.
    # ELI5: Your one question.
    messages=[{"role": "user", "content": "Explain what an AI agent is in 2 lines"}],
)
# WHY (tech): Reads the first choice's message text from the response.
# ELI5: Show the answer.
print(response.choices[0].message.content)

In [ ]:
# CHAT WITH MEMORY - turn 1 (Responses API)
# WHY (tech): The Responses API stores each response on OpenAI's servers with an ID for later linking.
# ELI5: OpenAI keeps the conversation in its own notebook and gives you a ticket number.
response1 = openai_client.responses.create(model=MODEL, input="Hi! My name is Satvik and I teach AI.")
# WHY (tech): output_text is the Responses API shortcut for the answer text.
# ELI5: Show the reply.
print(response1.output_text)

In [ ]:
# CHAT WITH MEMORY - turn 2
# WHY (tech): previous_response_id makes OpenAI load the earlier conversation server-side, so no history list is needed.
# ELI5: Show your ticket number; OpenAI pulls out the old conversation.
response2 = openai_client.responses.create(
    model=MODEL,
    # WHY (tech): Only the new message is sent; context comes from the linked response.
    # ELI5: Just ask the new question.
    input="What's my name, and what do I do?",
    # WHY (tech): Links to turn 1, so the model sees that exchange as context.
    # ELI5: Ticket number for the earlier chat.
    previous_response_id=response1.id,
)
# WHY (tech): Prints the answer, which should recall facts from turn 1.
# ELI5: Show the reply that proves it remembered.
print(response2.output_text)

In [ ]:
# CHAT WITH MEMORY - turn 3
# WHY (tech): Links to response2 (the latest turn), so the chain includes turns 1 and 2.
# ELI5: Always show the newest ticket so nothing is skipped.
response3 = openai_client.responses.create(
    model=MODEL,
    # WHY (tech): New user question; earlier context arrives through the linked response chain.
    # ELI5: Ask your next question.
    input="I have 5 students suggest me one thing to increase  my no of students?",
    # FIXED: was response1.id, which skipped turn 2. Always link to the most recent response.
    # WHY (tech): Each response links to its parent, forming a chain; linking to the latest keeps full history.
    # ELI5: Use the latest ticket, not an old one.
    previous_response_id=response2.id,
)
# WHY (tech): Prints the third answer.
# ELI5: Show the reply.
print(response3.output_text)

Antropic Claud Chatbot

In [ ]:
# WHY (tech): Imports Anthropic's SDK, which handles Claude's Messages API format and auth headers.
# ELI5: Bring in Anthropic's phone.
import anthropic
# WHY (tech): Imports Colab's secrets reader so the key stays out of the code.
# ELI5: Get the key to your password drawer.
from google.colab import userdata

# WHY (tech): Authenticated client; Claude API billing is separate from a Claude Pro subscription.
# ELI5: Call Claude and show your ID card.
claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
# WHY (tech): Model ID in one variable; overwrites the previous platform's MODEL value.
# ELI5: Write the Claude brain's name on the sticky note.
MODEL = "claude-sonnet-5-5"
# WHY (tech): Tiny test request; max_tokens is required by Claude and caps reply length.
# ELI5: Say "hello" once, with a short word limit.
test = claude.messages.create(model=MODEL, max_tokens=20, messages=[{"role": "user", "content": "Say OK"}])
# WHY (tech): [-1] takes the last content block, which holds the text even if a thinking block comes first.
# ELI5: Read the last page of the reply, where the answer is.
print("✅ Claude connected:", test.content[-1].text)

In [ ]:
# SINGLE QUESTION (no memory)
# WHY (tech): Messages API call with one user turn; stateless, so nothing is remembered afterwards.
# ELI5: Ask one question to someone who forgets you straight after.
response = claude.messages.create(
    # WHY (tech): Which Claude model processes the request.
    # ELI5: Which brain answers.
    model=MODEL,
    # WHY (tech): Required by Claude; maximum output tokens, which also limits cost.
    # ELI5: Most words Claude may write.
    max_tokens=300,
    # WHY (tech): Same role/content format as OpenAI.
    # ELI5: Your one question.
    messages=[{"role": "user", "content": "Explain what an AI agent is in 2 lines"}],
)
# WHY (tech): Last content block holds the answer text, even when thinking blocks precede it.
# ELI5: Show the answer.
print(response.content[-1].text)

Note: Claude and Groq: their SDKs have no chat object, so you create a small one once, and after that each message is a single line like Gemini’s.

In [ ]:
# OPTION A, version 1: helper function (superseded by the next cell; keep for learning)
# WHY (tech): Empty list that will store every turn; Claude is stateless, so we keep history ourselves.
# ELI5: A blank notebook for the conversation.
claude_history = []

# WHY (tech): A function wraps the add-send-save steps so each new message is one line.
# ELI5: A machine: put in a message, get an answer, notebook updated.
def claude_send(msg):
    # WHY (tech): Appends the user's turn to the history list.
    # ELI5: Write your message in the notebook.
    claude_history.append({"role": "user", "content": msg})
    # WHY (tech): Sends the entire history, so Claude sees all previous turns.
    # ELI5: Show Claude the whole notebook.
    r = claude.messages.create(model="claude-sonnet-5-5", max_tokens=500, messages=claude_history)
    # WHY (tech): Saves Claude's reply; content[0] can fail if a thinking block comes first (fixed below).
    # ELI5: Write Claude's answer in the notebook.
    claude_history.append({"role": "assistant", "content": r.content[0].text})
    # WHY (tech): Returns the answer text to the caller.
    # ELI5: Hand back the answer.
    return r.content[0].text

In [ ]:
# OPTION A, version 2: handles modern content blocks (such as ThinkingBlock)
# WHY (tech): Redefining the function replaces version 1; the same claude_history list is reused.
# ELI5: Upgrade the machine, keep the same notebook.
def claude_send(msg):
    # WHY (tech): Appends the user's turn to the history list.
    # ELI5: Write your message in the notebook.
    claude_history.append({"role": "user", "content": msg})
    # WHY (tech): Sends full history; larger max_tokens leaves room for thinking plus the answer.
    # ELI5: Show Claude the notebook, with a bigger word limit.
    r = claude.messages.create(model="claude-sonnet-5-5", max_tokens=1000, messages=claude_history)
    # WHY (tech): next() returns the first block that has .text; "" is a fallback so it never crashes.
    # ELI5: Flip through the reply pages and find the one with the answer.
    text_content = next((block.text for block in r.content if hasattr(block, 'text')), "")
    # WHY (tech): Saves the reply as "assistant" so Claude sees it on the next turn.
    # ELI5: Write Claude's answer in the notebook.
    claude_history.append({"role": "assistant", "content": text_content})
    # WHY (tech): Returns the answer text to the caller.
    # ELI5: Hand back the answer.
    return text_content

# WHY (tech): First turn of the conversation using the helper function.
# ELI5: Introduce yourself to Claude.
print(claude_send("Hi! My name is Satvik and I teach AI."))

In [ ]:
# WHY (tech): Second turn; the helper resends the stored history, so Claude can recall turn 1.
# ELI5: Ask a follow-up; Claude remembers because the notebook is shown again.
print(claude_send("What's my name, and what do I do?"))

In [ ]:
# OPTION B: simpler code, no function - turn 1
# WHY (tech): One stateless call; we keep response1 so its text can be resent in turn 2.
# ELI5: Ask once and keep the answer on paper.
response1 = claude.messages.create(
    model="claude-sonnet-5-5",
    # WHY (tech): Required output cap; 1000 leaves room for any thinking plus the answer.
    # ELI5: Word limit for Claude.
    max_tokens=1000,
    # WHY (tech): Single user turn in role/content format.
    # ELI5: Your first message.
    messages=[{"role": "user", "content": "Hi! My name is Satvik and I teach AI."}],
)
# WHY (tech): [-1] means the last block, which holds the answer text.
# ELI5: Read the last page, where the answer is.
print(response1.content[-1].text)

In [ ]:
# OPTION B: simpler code, no function - turn 2
# WHY (tech): We manually resend turn 1 (question and answer) because Claude keeps no memory between calls.
# ELI5: Remind Claude of the earlier chat, then ask the new question.
response2 = claude.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=1000,
    messages=[
        # WHY (tech): Turn 1 user message, repeated exactly as originally sent.
        # ELI5: What you said first.
        {"role": "user", "content": "Hi! My name is Satvik and I teach AI."},
        # WHY (tech): Turn 1 reply with role "assistant", taken from response1.
        # ELI5: What Claude said back.
        {"role": "assistant", "content": response1.content[-1].text},
        # WHY (tech): The new user message that Claude answers now.
        # ELI5: Your new question.
        {"role": "user", "content": "Discuss AI Ethics?"},
    ],
)
# WHY (tech): Prints the answer to the newest message.
# ELI5: Show the reply.
print(response2.content[-1].text)

Groq Chat Bot

In [ ]:
# WHY (tech): Groq is OpenAI-compatible, so we reuse OpenAI's SDK instead of installing another library.
# ELI5: Groq speaks OpenAI's language, so use the same phone.
from openai import OpenAI
# WHY (tech): Imports Colab's secrets reader so the key stays out of the code.
# ELI5: Get the key to your password drawer.
from google.colab import userdata

# WHY (tech): base_url redirects every request to Groq's servers; that one setting switches the provider.
# ELI5: Same phone, different phone number, Groq's ID card.
groq = OpenAI(api_key=userdata.get("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")
# WHY (tech): A chat-capable model from Groq's list; guard, whisper and TTS models can't chat.
# ELI5: Pick a brain that can actually chat.
MODEL = "openai/gpt-oss-120b"
# WHY (tech): Tiny test request confirms the key and model work.
# ELI5: Say "hello" once to check the line works.
test = groq.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Say OK"}])
# WHY (tech): Same response structure as OpenAI: first choice, then message text.
# ELI5: Read the reply to confirm you're connected.
print("✅ Groq connected:", test.choices[0].message.content)

In [ ]:
# SINGLE QUESTION (no memory)
# WHY (tech): Identical to the OpenAI call; only the client object differs.
# ELI5: Same order form, different shop.
response = groq.chat.completions.create(
    # WHY (tech): Which hosted model processes the request.
    # ELI5: Which brain answers.
    model=MODEL,
    # WHY (tech): Single user turn in role/content format.
    # ELI5: Your one question.
    messages=[{"role": "user", "content": "Explain what an AI agent is in 2 lines"}],
)
# WHY (tech): Reads the first choice's message text.
# ELI5: Show the answer.
print(response.choices[0].message.content)

In [ ]:
# OPTION A: helper function (run once)
# WHY (tech): Empty list to store turns; Groq is stateless, so history lives in our notebook.
# ELI5: A blank notebook for the conversation.
groq_history = []

# WHY (tech): Function wraps add-send-save steps, so each new message is a single line.
# ELI5: A machine: put in a message, get an answer, notebook updated.
def groq_send(msg):
    # WHY (tech): Appends the user's turn to the history list.
    # ELI5: Write your message in the notebook.
    groq_history.append({"role": "user", "content": msg})
    # WHY (tech): Sends the full history so the model sees every earlier turn.
    # ELI5: Show Groq the whole notebook.
    r = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=groq_history)
    # WHY (tech): Saves the reply as "assistant" so it is included next time.
    # ELI5: Write Groq's answer in the notebook.
    groq_history.append({"role": "assistant", "content": r.choices[0].message.content})
    # WHY (tech): Returns the answer text to the caller.
    # ELI5: Hand back the answer.
    return r.choices[0].message.content

In [ ]:
# WHY (tech): First turn using the helper; history now holds one question and one answer.
# ELI5: Introduce yourself to Groq.
print(groq_send("Hi! My name is Satvik and I teach AI."))

In [ ]:
# WHY (tech): Second turn; the helper resends stored history, so the model can recall turn 1.
# ELI5: Ask a follow-up; Groq remembers because the notebook is shown again.
print(groq_send("What's my name, and what do I do?"))

Note : Groq’s code is the same as OpenAI’s. The only differences are the connection name (groq) and the model name. Compared with Claude, Groq doesn’t need max_tokens, and you read the answer with choices[0].message.content instead of content[-1].text.

In [ ]:
# OPTION B: simpler code, no function - turn 1
# WHY (tech): One stateless call; we keep response1 so its text can be resent in turn 2.
# ELI5: Ask once and keep the answer on paper.
response1 = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    # WHY (tech): Single user turn in role/content format.
    # ELI5: Your first message.
    messages=[{"role": "user", "content": "Hi! My name is Satvik and I teach AI."}],
)
# WHY (tech): Reads the first choice's message text.
# ELI5: Show the reply.
print(response1.choices[0].message.content)

In [ ]:
# OPTION B: simpler code, no function - turn 2
# WHY (tech): Manually resends turn 1 because the API keeps no memory between calls.
# ELI5: Remind Groq of the earlier chat, then ask the new question.
response2 = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        # WHY (tech): Turn 1 user message, repeated exactly as originally sent.
        # ELI5: What you said first.
        {"role": "user", "content": "Hi! My name is Satvik and I teach AI."},
        # WHY (tech): Turn 1 reply with role "assistant", taken from response1.
        # ELI5: What Groq said back.
        {"role": "assistant", "content": response1.choices[0].message.content},
        # WHY (tech): The new user message the model answers now.
        # ELI5: Your new question.
        {"role": "user", "content": "What's my name, and what do I do?"},
    ],
)
# WHY (tech): Prints the answer to the newest message.
# ELI5: Show the reply.
print(response2.choices[0].message.content)